# Auxiliary-loss-free load balancing

This notebook demonstrates expert-selection biases, measured load updates, and a controlled balancing experiment.

In [ ]:
import sys
from pathlib import Path

import torch

project_root = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate / "src" / "model").is_dir()),
    None,
)
if project_root is None:
    raise RuntimeError("Could not find the repository root")
sys.path.insert(0, str(project_root))
torch.manual_seed(18)

In [ ]:
from src.model import DeepSeekConfig, TopKRouter

config = DeepSeekConfig(dim=64, n_routed_experts=4, n_activated_experts=2)
router = TopKRouter(config)
indices = torch.tensor([[0, 1], [0, 1], [0, 1], [0, 1], [0, 2], [0, 3]])
counts = router.update_bias(indices, gamma=0.05)
print("Counts:", counts.tolist())
print("Biases:", router.expert_bias.tolist())

In [ ]:
def load_metrics(counts):
    counts = counts.float()
    mean = counts.mean()
    if mean.item() == 0:
        return 0.0, 0.0
    cv = (counts.std(unbiased=False) / mean).item()
    max_ratio = (counts.max() / mean).item()
    return cv, max_ratio

config = DeepSeekConfig(dim=64, n_routed_experts=8, n_activated_experts=2)
router = TopKRouter(config).eval()
tokens = torch.randn(128, config.dim) * 0.1
tokens[:, 0] = 1.0
with torch.no_grad():
    router.gate.weight.zero_()
    router.gate.weight[0, 0] = 2.0
    router.gate.weight[1, 0] = 1.5

for step in range(21):
    with torch.no_grad():
        indices, _ = router(tokens)
    counts = torch.bincount(indices.reshape(-1), minlength=config.n_routed_experts)
    cv, max_ratio = load_metrics(counts)
    if step % 5 == 0:
        print(step, counts.tolist(), round(cv, 3), round(max_ratio, 3))
    router.update_bias(indices, gamma=0.01)

In [ ]:
x = torch.randn(2, 4, config.dim, requires_grad=True)
router = TopKRouter(config)
_, weights = router(x)
weights[..., 0].sum().backward()
print("Gate gradient exists:", router.gate.weight.grad is not None)
print("Bias gradient:", router.expert_bias.grad)